In [2]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

transactions = [
    ["bread", "milk", "butter"],
    ["bread", "milk"],
    ["bread", "eggs"],
    ["milk", "eggs"],
    ["bread", "milk", "eggs"],
    ["bread", "butter"],
    ["milk", "butter"],
    ["bread", "milk", "butter", "eggs"],
    ["milk", "eggs", "butter"],
    ["bread", "milk", "eggs"],
    ["bread", "butter"],
    ["milk", "butter"],
    ["bread", "milk"],
    ["bread", "eggs", "butter"],
    ["milk", "eggs"],
]

len(transactions)

15

In [3]:
te = TransactionEncoder()
encoded_data = te.fit_transform(transactions)

basket = pd.DataFrame(
    encoded_data,
    columns = te.columns_
)

frequent_itemsets = apriori(
    basket,
    min_support = 0.2,
    use_colnames = True
)
frequent_itemsets

,support,itemsets
0,0.666667,frozenset({bread})
1,0.533333,frozenset({butter})
2,0.533333,frozenset({eggs})
3,0.733333,frozenset({milk})
4,0.333333,"frozenset({butter, bread})"
5,0.333333,"frozenset({eggs, bread})"
6,0.400000,"frozenset({milk, bread})"
7,0.200000,"frozenset({butter, eggs})"
8,0.333333,"frozenset({butter, milk})"
9,0.400000,"frozenset({eggs, milk})"


In [4]:
rules = association_rules(
    frequent_itemsets,
    metric = "confidence",
    min_threshold = 0.5
)

rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]]

,antecedents,consequents,support,confidence,lift
0,frozenset({butter}),frozenset({bread}),0.333333,0.625000,0.937500
1,frozenset({bread}),frozenset({butter}),0.333333,0.500000,0.937500
2,frozenset({eggs}),frozenset({bread}),0.333333,0.625000,0.937500
3,frozenset({bread}),frozenset({eggs}),0.333333,0.500000,0.937500
4,frozenset({milk}),frozenset({bread}),0.400000,0.545455,0.818182
5,frozenset({bread}),frozenset({milk}),0.400000,0.600000,0.818182
6,frozenset({butter}),frozenset({milk}),0.333333,0.625000,0.852273
7,frozenset({eggs}),frozenset({milk}),0.400000,0.750000,1.022727
8,frozenset({milk}),frozenset({eggs}),0.400000,0.545455,1.022727
9,"frozenset({eggs, milk})",frozenset({bread}),0.200000,0.500000,0.750000


In [5]:
import mlflow
from pathlib import Path

project_dir = Path.cwd().parent
db_path = project_dir / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{db_path.as_posix()}")
mlflow.set_experiment("Market Basket Association Rules")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment created successfully")

c:\Users\Mumba Mumba\Desktop\association-rule-mlops\.venv\Lib\site-packages\mlflow\store\tracking\sqlalchemy_store.py:327: SADeprecationWarning: The ``noload`` loader strategy is deprecated and will be removed in a future release.  This option produces incorrect results by returning ``None`` for related items. (deprecated since: 2.1)
  sqlalchemy.orm.configure_mappers()


Tracking URI: sqlite:///c:/Users/Mumba Mumba/Desktop/association-rule-mlops/mlflow.db
Experiment created successfully


In [6]:
with mlflow.start_run(run_name="apriori_baseline"):

    min_support = 0.2
    min_confidence = 0.5

    itemsets = apriori(
        basket,
        min_support=min_support,
        use_colnames=True
    )

    rules = association_rules(
        itemsets,
        metric="confidence",
        min_threshold=min_confidence
    )

    mlflow.log_param("algorithm", "Apriori")
    mlflow.log_param("min_support", min_support)
    mlflow.log_param("min_confidence", min_confidence)

    mlflow.log_metric("number_of_itemsets", len(itemsets))
    mlflow.log_metric("number_of_rules", len(rules))

    if not rules.empty:
        mlflow.log_metric("average_lift", rules["lift"].mean())

    rules.to_csv("rules_baseline.csv", index=False)
    mlflow.log_artifact("rules_baseline.csv")

    print("Baseline itemsets:", len(itemsets))
    print("Baseline rules:", len(rules))

Baseline itemsets: 11
Baseline rules: 12


In [7]:
with mlflow.start_run(run_name="apriori_stricter"):

    min_support = 0.3
    min_confidence = 0.7

    itemsets_2 = apriori(
        basket,
        min_support=min_support,
        use_colnames=True
    )

    rules_2 = association_rules(
        itemsets_2,
        metric="confidence",
        min_threshold=min_confidence
    )

    mlflow.log_param("algorithm", "Apriori")
    mlflow.log_param("min_support", min_support)
    mlflow.log_param("min_confidence", min_confidence)

    mlflow.log_metric("number_of_itemsets", len(itemsets_2))
    mlflow.log_metric("number_of_rules", len(rules_2))

    if not rules_2.empty:
        mlflow.log_metric("average_lift", rules_2["lift"].mean())

    rules_2.to_csv("rules_stricter.csv", index=False)
    mlflow.log_artifact("rules_stricter.csv")

    print("Stricter itemsets:", len(itemsets_2))
    print("Stricter rules:", len(rules_2))

Stricter itemsets: 9
Stricter rules: 1


In [8]:
import mlflow

print("Tracking URI:", mlflow.get_tracking_uri())

experiment = mlflow.get_experiment_by_name(
    "Market Basket Association Rules"
)

print("Experiment:", experiment)

Tracking URI: sqlite:///c:/Users/Mumba Mumba/Desktop/association-rule-mlops/mlflow.db
Experiment: <Experiment: artifact_location='file:c:/Users/Mumba Mumba/Desktop/association-rule-mlops/notebooks/mlruns/1', creation_time=1792216057519, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792216057519, lifecycle_stage='active', name='Market Basket Association Rules', tags={}, trace_location=None, workspace='default'>


In [9]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

runs[["run_id", "tags.mlflow.runName", "status"]]


,run_id,tags.mlflow.runName,status
0,b71e3de4db1e457e9b138150056f991f,apriori_stricter,FINISHED
1,0a0c241e38114a208f1fd37a6b5b53aa,apriori_baseline,FINISHED
2,826fa2074d04453e9774ef0821c67f89,apriori_stricter,FINISHED
3,c1d4fb88a5804139b6a09577f2b33493,apriori_baseline,FINISHED
4,4a5846e825ac42019c00df110b03770a,apriori_stricter,FINISHED
5,7555a876907d484db6b1eb2a7978c093,apriori_stricter,FAILED
6,b10e496cacba46139736d1b003766a68,apriori_baseline,FINISHED
7,31e9801009a440c8bffa95bad5ab5b4d,apriori_baseline,FINISHED


In [10]:
from pathlib import Path

print(Path("rules_baseline.csv").resolve())

C:\Users\Mumba Mumba\Desktop\association-rule-mlops\notebooks\rules_baseline.csv
